# Q-learning (model-free)

_Artificial Intelligence (CS221)_

**No map of the world? Learn the value of actions by trying them and seeing what happens.**

Value iteration needs the transition probabilities $T$. Often we do not have them.

     Q-learning learns Q-values directly from experience. No model needed.

---

This notebook is a practice scaffold for the **Q-learning (model-free)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# Chain: states 0..3, state 3 is the goal. Action 0 = left, 1 = right.
N, A = 4, 2
def step(s, a):
    s2 = max(0, s - 1) if a == 0 else min(N - 1, s + 1)
    reward = 10.0 if s2 == N - 1 else -1.0
    return s2, reward

Q = np.zeros((N, A))
alpha, gamma, eps = 0.5, 0.9, 0.2

for ep in range(400):
    s = 0
    for _ in range(20):
        a = rng.integers(A) if rng.random() < eps else int(Q[s].argmax())
        s2, r = step(s, a)
        # Q-learning update toward r + gamma * best next Q
        Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
        s = s2
        if s == N - 1:
            break

print("learned Q-table:\n", np.round(Q, 2))
print("greedy policy (0=left,1=right):", Q.argmax(axis=1))

## Visualize the data & results

_Can a warehouse robot learn its way to the goal shelf by trial and error, never told the floor layout?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ROWS, COLS = 3, 4
WALL, GOAL, HAZARD = (1,1), (0,3), (1,3)
gamma = 0.95
acts = [(-1,0),(1,0),(0,-1),(0,1)]      # up, down, left, right
def ok(s): return 0<=s[0]<ROWS and 0<=s[1]<COLS and s != WALL
states = [(r,c) for r in range(ROWS) for c in range(COLS) if (r,c)!=WALL]
def step(s, ai):
    d = acts[ai]; ns = (s[0]+d[0], s[1]+d[1])
    if not ok(ns): ns = s
    if ns == GOAL: return ns, 1.0, True
    if ns == HAZARD: return ns, -1.0, True
    return ns, -0.04, False

rng = np.random.default_rng(1)
Q = {(s, a): 0.0 for s in states for a in range(4)}
alpha, eps = 0.5, 0.2
for ep in range(3000):
    s = (2, 0)                          # charging dock
    for t in range(50):
        greedy = int(np.argmax([Q[(s,i)] for i in range(4)]))
        a = rng.integers(4) if rng.random() < eps else greedy
        ns, r, done = step(s, a)
        nxt = 0 if done else max(Q[(ns,i)] for i in range(4))
        Q[(s,a)] += alpha * (r + gamma*nxt - Q[(s,a)])
        s = ns
        if done: break

grid = np.full((ROWS, COLS), np.nan)
for s in states: grid[s] = max(Q[(s,i)] for i in range(4))
grid[GOAL] = 1.0; grid[HAZARD] = -1.0
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(grid, cmap="viridis")
for r in range(ROWS):
    for c in range(COLS):
        if (r,c) != WALL:
            ax.text(c, r, round(grid[r,c], 2), ha="center", va="center", color="white")
ax.set_title("warehouse floor: best learned Q per cell")
fig.colorbar(im, ax=ax); plt.show()